# sec-rag — driver notebook

Thin driver: builds one `SecRag` (all setup lives in its `__init__`), then uses
`rag.search(...)`, the eval harness, and `rag.run_agent(...)`. All logic lives in
`src/sec_rag/`.

In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
from sec_rag.core import SecRag

# SecRag.__init__ does ALL setup, once: load the persisted Chroma vector store from
# disk, re-ingest the Apple 10-K to rebuild fixed_chunks, build the BM25 index, load
# the CrossEncoder reranker, create the Anthropic client (key from .env), load the
# rules prompt. Exposed as rag.vectorestore / rag.bm25 / rag.fixed_chunks /
# rag.reranker / rag.client / rag.rules.
rag = SecRag()

## One-time bootstrap — how the vector store was first populated

**Already done — do not re-run.** The cell below is the original ingestion that
built `vectorStore/` on disk (embed every chunk with `all-MiniLM-L6-v2`, write the
parallel lists into a persistent Chroma collection). It is kept only as a record.
`SecRag()` above loads that persisted store; it never re-embeds. Re-run this only to
rebuild the store from scratch (e.g. after deleting `vectorStore/`).

In [ ]:
# === ONE-TIME BOOTSTRAP (already run) — builds vectorStore/ from scratch ===========
# Left unexecuted on purpose. This is the exact sequence that first populated the
# persistent Chroma store SecRag() now loads. Run from the repo root so the
# relative path below resolves.
#
# import os
# from edgar import set_identity, Company
# from langchain_text_splitters import RecursiveCharacterTextSplitter
# from sentence_transformers import SentenceTransformer
# import chromadb
# from sec_rag.ingest import process_filing
# from sec_rag.embed import embedding
#
# # 1. Fetch + parse Apple's most recent 10-K
# set_identity(os.getenv("SEC_IDENTITY", "sec-rag example@example.com"))
# tenk = Company("AAPL").get_filings(form="10-K")[0].obj()
#
# # 2. Chunk it
# splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
# fixed_chunks = process_filing(tenk, splitter)
#
# # 3. Embed every chunk with all-MiniLM-L6-v2
# embedder = SentenceTransformer("all-MiniLM-L6-v2")
# embedded_chunks = []
# for chunk in fixed_chunks:
#     chunk["embedding"] = embedding(chunk["text"], embedder)
#     embedded_chunks.append(chunk)
#
# # 4. Write the parallel lists into a persistent Chroma collection
# chroma_client = chromadb.PersistentClient(path="vectorStore")
# vectorestore = chroma_client.get_or_create_collection(name="vectorStore")
# ids = [f"{c['Company']}_{c['item']}_{i}" for i, c in enumerate(embedded_chunks)]
# vectorestore.add(
#     ids=ids,
#     embeddings=[c["embedding"] for c in embedded_chunks],
#     documents=[c["text"] for c in embedded_chunks],
#     metadatas=[{"Company": c["Company"], "Period": c["Period of report"], "Form": c["Form"]}
#                for c in embedded_chunks],
# )

## Try it

In [ ]:
rag.search("What was Apple's total revenue in 2025?")

## Run the eval

In [ ]:
from sec_rag.evaluate import judge_rules, eval_set, run_eval

scored = run_eval(
    eval_set,
    rag.vectorestore, rag.bm25, rag.fixed_chunks, rag.reranker, rag.client, rag.rules,
    judge_rules,
)

# Show only the failed questions, so you can inspect what went wrong
[s for s in scored if not s["passed"]]

In [ ]:
from sec_rag.retrieve import retrieve, rerank

In [ ]:
q = "How much did Apple spend on research and development in 2025?"
candidates = retrieve(q, rag.vectorestore, rag.bm25, rag.fixed_chunks)
ranked = rerank(q, candidates, rag.reranker)

for c in rag.fixed_chunks:
    if "34,550" in c["text"]:
        print("FOUND $34,550 in:", c["text"][:150])

## Agent

`rag.run_agent(question)` runs the tool-use loop (tools: `calculator`,
`search_filings` → `rag.search`). Returns `(answer, steps)`, where each step is
`{"tool", "input", "result"}`. `agent_eval` below is reference data for multi-step
questions.

### AI AGENT EVAL

In [ ]:
agent_eval = [
    {
        "question": "What was Apple's R&D spending as a percentage of revenue in 2025?",
        "expected_answer": "About 8.3% ($34,550M R&D / $416,161M revenue)",
        "expected_steps": ["look up R&D spending", "look up total revenue", "compute ratio"],
    },
    {
        "question": "How much did Apple's net income change from 2024 to 2025?",
        "expected_answer": "Up about $18,274M (from $93,736M to $112,010M)",
        "expected_steps": ["look up 2025 net income", "look up 2024 net income", "compute difference"],
    },
    {
        "question": "Which of Apple's operating expense lines was larger in 2025, R&D or SG&A, and by how much?",
        "expected_answer": "R&D was larger by about $6,949M ($34,550M vs $27,601M)",
        "expected_steps": ["look up R&D", "look up SG&A", "compare / subtract"],
    },
    {
        "question": "What was Apple's total operating expense in 2025, combining R&D and SG&A?",
        "expected_answer": "About $62,151M ($34,550M + $27,601M)",
        "expected_steps": ["look up R&D", "look up SG&A", "sum them"],
    },
    {
        "question": "How much did Apple's R&D spending grow from 2023 to 2025 in absolute terms?",
        "expected_answer": "Up about $4,635M (from $29,915M to $34,550M)",
        "expected_steps": ["look up 2025 R&D", "look up 2023 R&D", "compute difference"],
    },
]

In [ ]:
from sec_rag.evaluate import judge_the_eval
from sec_rag.evaluate import judge_rules_agent

In [ ]:
results = []
for item in agent_eval:
    answer, steps, usage = rag.run_agent(item["question"])
    tools_used = [s["tool"] for s in steps]          # the actual path

    verdict = judge_the_eval(item["expected_answer"], answer, item["question"], rag.client, judge_rules_agent)  # your existing judge

    results.append({
        "question": item["question"],
        "answer": answer,
        "correct": verdict,
        "expected_steps": item["expected_steps"],
        "tools_used": tools_used,
        "num_steps": len(steps),
        "tokens": usage["input_tokens"] + usage["output_tokens"],
    })

results

### RAPTOR 

In [ ]:
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
import numpy as np
print("all imported")

In [ ]:
print(rag.fixed_chunks[0])

In [ ]:
from sec_rag.embed import embedding

for chunk in rag.fixed_chunks:
    chunk["embedding"] = embedding(chunk["text"], rag.embedder)

print(rag.fixed_chunks[0])

In [ ]:
import numpy as np

embeddings = np.array([chunk["embedding"] for chunk in rag.fixed_chunks])
print(embeddings.shape)

In [ ]:

reducer = PCA(n_components=10)
reduced = reducer.fit_transform(embeddings)
print(reduced)

In [ ]:
from sklearn.mixture import GaussianMixture
import numpy as np

n_range = range(2, 15)          # try 2 up to 14 clusters
bics = []
for n in n_range:
    gmm = GaussianMixture(n_components=n, random_state=42)
    gmm.fit(reduced)
    bics.append(gmm.bic(reduced))   # score this cluster count

best_n = n_range[np.argmin(bics)]   # the count with the lowest (best) BIC
print("best number of clusters:", best_n)

In [ ]:
gmm = GaussianMixture(n_components=best_n, random_state=42)
gmm.fit(reduced)
probs = gmm.predict_proba(reduced)
print(probs)

In [ ]:
import numpy as np
# how many chunks have a "split" membership (belong to 2+ clusters above 0.1)?
above = (probs >= 0.10).sum(axis=1)   # per chunk: how many clusters it's in
print("chunks in 1 cluster:", (above == 1).sum())
print("chunks in 2+ clusters:", (above >= 2).sum())
print("max second-highest prob:", np.sort(probs, axis=1)[:, -2].max())

In [ ]:
threshold = 0.10
clusters = {i: [] for i in range(best_n)}   # {0: [], 1: [], 2: []}

for chunk_idx, row in enumerate(probs):
    for cluster_idx, prob in enumerate(row):
        if prob >= threshold:
            clusters[cluster_idx].append(rag.fixed_chunks[chunk_idx])

for i in range(best_n):
    print(f"cluster {i}: {len(clusters[i])} chunks")

In [ ]:
def summarize_cluster(chunks, client):
    # join all the chunk texts in this cluster into one big blob
    combined = "\n\n".join(c["text"] for c in chunks)

    prompt = f"""Summarize the following excerpts from Apple's 2025 10-K filing into a concise overview that captures the main topics, facts, and figures. Keep it factual and grounded in the text.

{combined}

Summary:"""

    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=1024,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.content[0].text

In [ ]:
summaries = []
for i in range(best_n):
    summary_text = summarize_cluster(clusters[i], rag.client)
    summaries.append(summary_text)
    print(f"--- Cluster {i} summary ---")
    print(summary_text[:300])
    print()

In [ ]:
from sec_rag.retrieve import retrieve, rerank

In [ ]:
print(rag.search("what does apple even make"))